# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from groq import Groq

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama-3.3-70b-versatile'
groq = Groq()

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have Llama-3.3-70b-versatile figure out which links are relevant

### Use a call to llama-3.3-70b-versatile to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [8]:
def select_relevant_links(url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to Groq Llama-3.3-70b-versatile

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
2 days ago
•
174k
•
3.15k
openai/privacy-filter
Updated
6 days ago
•
57.7k
•
1.03k
Qwen/Qwen3.6-27B
Updated
5 days ago
•
509k
•
970
deepseek-ai/DeepSeek-V4-Flash
Updated
2 days ago
•
96.9k
•
824
unsloth/Qwen3.6-27B-GGUF
Updated
6 days ago
•
702k
•
480
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
241
ML Intern
🤖
241
Explore ML concepts and get help with a virtual intern
Running
on
Zero
MCP
2.43k
Wan2.2 14B Preview
🐌
2.43k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
1.05k
FireRed Image Edit 1.0 Fast
🌖
1.05k
FireRed-Image-Edit × Qwen-Image-Edit-Rapid (Transf

In [24]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n2 days ago\n•\n174k\n•\n3.15k\nopenai/privacy-filter\nUpdated\n6 days ago\n•\n57.7k\n•\n1.03k\nQwen/Qwen3.6-27B\nUpdated\n5 days ago\n•\n509k\n•\n970\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\n2 days ago\n•\n96.9k\n•\n825\nunsloth/Qwen3.6-27B-GGUF\nUpdated\n6 days ago\n•\n702k\n•\n480\nBrowse 2M+ models\nSpaces\nRunning\non\nCPU Upgrade\n241\nML Intern

In [27]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [28]:
create_brochure("HuggingFace", "https://huggingface.co")

# Welcome to Hugging Face: The AI Community Building the Future
Hugging Face is the ultimate platform where machine learning enthusiasts, researchers, and developers collaborate on models, datasets, and applications. With over 2 million models, 1 million applications, and 500,000 datasets, our community is the largest and most vibrant in the AI space.

## Our Culture
At Hugging Face, we're passionate about open-source collaboration, innovation, and community-driven growth. Our platform is designed to foster creativity, experimentation, and knowledge-sharing among AI enthusiasts. We believe in making AI more accessible, transparent, and beneficial for everyone.

## Our Community
Our community is at the heart of everything we do. From researchers to developers, our members are pushing the boundaries of what's possible with AI. With features like model sharing, dataset collaboration, and application development, our platform provides the perfect environment for like-minded individuals to come together and create something amazing.

## Careers at Hugging Face
We're always looking for talented individuals to join our team. As a Hugging Face employee, you'll have the opportunity to work on cutting-edge AI projects, collaborate with leading researchers, and contribute to the growth of our community. Whether you're a software engineer, researcher, or just starting your career, we have a range of exciting opportunities available.

## Join the Future of AI
At Hugging Face, we're committed to building a better future for everyone. By providing a platform for collaboration, innovation, and knowledge-sharing, we're empowering the next generation of AI leaders to create, discover, and push the boundaries of what's possible. Join us today and become a part of the AI community that's shaping the future.

## What Our Community Says
Our community is our greatest asset, and we're proud of the amazing work they're doing. From developing state-of-the-art models to creating innovative applications, our members are making a real impact in the AI space. Join us and become a part of this vibrant community of AI enthusiasts.

## Get Started
Ready to start your AI journey? Sign up for our platform today and gain access to our vast library of models, datasets, and applications. With Hugging Face, you'll be able to collaborate with others, learn from the best, and contribute to the growth of our community. Join us and become a part of the AI revolution.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from Groq,
with the familiar typewriter animation

In [31]:
def stream_brochure(company_name, url):
    stream = groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [32]:
stream_brochure("HuggingFace", "https://huggingface.co")

**The Hugging Face Brochure**

**Welcome to the Hugging Face Universe!**

Imagine a world where machine learning engineers, scientists, and end users come together to build a future where AI is open, accessible, and beneficial to all. Welcome to Hugging Face, the collaboration platform at the heart of the AI revolution!

**Our Mission**

Hugging Face empowers the next generation of machine learning professionals to learn, collaborate, and share their work to build an open and ethical AI future. With our fast-growing community, open-source libraries and tools, and a talented science team, we're changing the face of AI.

**What We Do**

* **Models**: Explore 2M+ pre-trained models for text, image, video, audio, and 3D applications.
* **Datasets**: Access 500k+ datasets to fuel your machine learning projects.
* **Spaces**: Discover and experiment with interactive AI applications on our platform.
* **Buckets**: Store and share your models, datasets, and applications in a secure and scalable environment.

**Our Community**

* 2M+ models shared by the community
* 500k+ datasets available for use
* 1M+ applications being experimented with in our Spaces features
* A vibrant community of 100k+ members on Discord

**Why Join Us**

* **Be part of a community**: Collaborate with like-minded individuals from around the world.
* **Get access to open-source libraries**: Leverage our pre-built models, datasets, and applications.
* **Stay up-to-date with the latest research**: Read our blog and explore the latest articles from our community.

**Careers at Hugging Face**

Join our team of passionate engineers, scientists, and experts in AI. We're always excited to meet talented individuals who share our vision for a more accessible and equitable AI future.

**How to Get Started**

Explore our platform to discover the power of Hugging Face. Browse our models, datasets, and applications, and join our community to start collaborating today!

**Connect with Us**

Twitter: @huggingface
LinkedIn: linkedin.com/company/hugging-face
GitHub: github.com/huggingface
Discord: discord.gg/huggingface

**Join the Hugging Face Universe today!**